# Duka Electronics Sales Analysis

## Executive Summary

- Nairobi was the strongest branch, generating KES 26,106,100 in revenue from 376 orders.
- Laptops were the highest-revenue product, contributing KES 41,535,000.
- Weekday sales accounted for 72.3% of orders and 72.23% of total revenue.
- April recorded the highest monthly revenue at KES 13,518,300, while June was the lowest at KES 10,903,400.
- The Premium Online filter for Nairobi and Mombasa identified 83 qualifying orders worth KES 10,218,000.

In [1]:
!python generate_data.py

All datasets generated successfully!
Duka sales: 1000 rows
Agents: 20 rows
Agent transactions: 612 rows


In [2]:
import pandas as pd
import numpy as np

sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

print("Dataset loaded successfully!")
print("Shape:", sales.shape)

sales.head()

Dataset loaded successfully!
Shape: (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500



## Task 1: Data Loading and First-Five-Minutes Inspection

The first stage of the analysis involves inspecting the Duka Electronics sales dataset to understand its structure, data types, completeness, date coverage and key categories before performing further analysis.


In [3]:

# Display dataset dimensions
print("Dataset Shape:", sales.shape)

# Display column names
print("\nColumn Names:")
print(sales.columns.tolist())

# Display data types and non-null counts
print("\nDataset Information:")
sales.info()


Dataset Shape: (1000, 7)

Column Names:
['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel', 'unit_price']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


In [4]:
# Check missing values in each column
print("Missing Values:")
print(sales.isnull().sum())

Missing Values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64


In [5]:
print("Earliest Order Date:", sales["order_date"].min())
print("Latest Order Date:", sales["order_date"].max())

Earliest Order Date: 2026-01-01 00:00:00
Latest Order Date: 2026-06-29 00:00:00


In [6]:
print("Branches:")
print(sales["branch"].value_counts())

print("\nProducts:")
print(sales["product"].value_counts())

print("\nSales Channels:")
print(sales["channel"].value_counts())

Branches:
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

Products:
product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64

Sales Channels:
channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64


In [7]:
sales.describe(include="all")

,order_id,order_date,branch,product,quantity,channel,unit_price
count,1000.000000,1000,1000,1000,1000.000000,1000,1000.000000
unique,NaN,NaN,5,5,NaN,3,NaN
top,NaN,NaN,Nairobi,Laptop,NaN,In-store,NaN
freq,NaN,NaN,376,220,NaN,579,NaN
mean,1500.500000,2026-03-30 08:44:09.600000,NaN,NaN,3.012000,NaN,23918.300000
min,1001.000000,2026-01-01 00:00:00,NaN,NaN,1.000000,NaN,1200.000000
25%,1250.750000,2026-02-14 00:00:00,NaN,NaN,2.000000,NaN,2500.000000
50%,1500.500000,2026-03-29 00:00:00,NaN,NaN,3.000000,NaN,15000.000000
75%,1750.250000,2026-05-14 00:00:00,NaN,NaN,4.000000,NaN,28000.000000
max,2000.000000,2026-06-29 00:00:00,NaN,NaN,5.000000,NaN,65000.000000



### First-Five-Minutes Inspection Findings

The Duka Electronics dataset contains 1,000 sales records and 7 columns. The dataset covers the six-month period from 1 January 2026 to 29 June 2026, which is suitable for the half-year business review.

The dataset contains no missing values across its seven columns, indicating that all records have complete entries.

The order_date column was parsed as datetime during loading. The remaining columns should be checked to confirm that their data types are appropriate for the planned analysis.

The dataset contains five branches: Nairobi, Mombasa, Kisumu, Nakuru and Eldoret. It also contains five product categories: Phone, Laptop, Headphones, Tablet and Charger.

Sales are recorded through three channels: In-store, Online and Phone order. These categories will allow comparisons of sales performance across branches, products and channels.


In [8]:
print(sales.dtypes)

order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object


In [9]:
print("Branches:", sorted(sales["branch"].unique()))
print("Products:", sorted(sales["product"].unique()))
print("Channels:", sorted(sales["channel"].unique()))

Branches: ['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']
Products: ['Charger', 'Headphones', 'Laptop', 'Phone', 'Tablet']
Channels: ['In-store', 'Online', 'Phone order']


## Task 2: Creating Derived Columns

Derived columns are created to support sales analysis. Revenue is calculated by multiplying quantity by unit price. Additional date-based columns identify the month, weekday and weekend orders, while price_band classifies products as Premium or Value based on their unit prices.

In [10]:

# Calculate total revenue for each order
sales["revenue"] = sales["quantity"] * sales["unit_price"]

# Extract the month name
sales["month"] = sales["order_date"].dt.month_name()

# Extract the weekday name
sales["weekday"] = sales["order_date"].dt.day_name()

# Identify weekend orders (Saturday = 5, Sunday = 6)
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5

# Classify products into Premium and Value
sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

# Display the updated dataset
sales.head()


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


In [11]:
print("Updated Dataset Shape:", sales.shape)

print("\nNew Columns:")
print(sales[[
    "revenue",
    "month",
    "weekday",
    "is_weekend",
    "price_band"
]].head())

print("\nPrice Band Distribution:")
print(sales["price_band"].value_counts())

Updated Dataset Shape: (1000, 12)

New Columns:
   revenue     month    weekday  is_weekend price_band
0    45000      June  Wednesday       False      Value
1   260000  February     Monday       False    Premium
2   140000   January     Monday       False    Premium
3    12500     April     Sunday        True      Value
4     7500     March   Saturday        True      Value

Price Band Distribution:
price_band
Value      565
Premium    435
Name: count, dtype: int64


In [12]:
sales[["quantity", "unit_price", "revenue"]].head(10)

,quantity,unit_price,revenue
0,3,15000,45000
1,4,65000,260000
2,5,28000,140000
3,5,2500,12500
4,3,2500,7500
5,1,2500,2500
6,2,65000,130000
7,5,1200,6000
8,5,2500,12500
9,5,1200,6000


## Task 3: Branch Performance Analysis

This task compares the performance of the five branches using two measures: total revenue generated and the number of orders received. The branches will be ranked separately to determine whether the branch receiving the most orders also generates the highest revenue.

In [13]:

# Calculate total revenue for each branch
branch_revenue = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Branch Ranking by Total Revenue:")
print(branch_revenue)


Branch Ranking by Total Revenue:
branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200
Name: revenue, dtype: int64


In [14]:

# Count orders received by each branch
branch_orders = (
    sales.groupby("branch")["order_id"]
    .count()
    .sort_values(ascending=False)
)

print("Branch Ranking by Number of Orders:")
print(branch_orders)


Branch Ranking by Number of Orders:
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: order_id, dtype: int64


In [15]:
branch_comparison = pd.DataFrame({
    "Total Revenue (KES)": branch_revenue,
    "Number of Orders": branch_orders
})

print("Branch Performance Comparison:")
display(branch_comparison)

Branch Performance Comparison:


,Total Revenue (KES),Number of Orders
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117



### Business Interpretation

The branch rankings are identical for both total revenue and number of orders. Nairobi ranks first, generating KES 26,106,100 from 376 orders, making it the strongest-performing branch. Mombasa ranks second with KES 14,003,700 from 186 orders, while Eldoret records the lowest performance with KES 8,844,200 from 117 orders.

The results suggest that branches receiving more orders generally generate higher total revenue. However, order volume alone does not determine revenue because the value of individual orders can differ depending on the products purchased and quantities sold.

Duka Electronics should maintain Nairobi's strong performance while investigating ways to increase sales in Eldoret and improve revenue generation across all branches.


## Task 4: Product Performance Analysis

This task identifies the top three products based on total revenue and total units sold. Comparing the two rankings will help Duka Electronics understand the difference between products that generate the most income and products that have the highest sales volume.

In [16]:
# Calculate total revenue by product
product_revenue = (
    sales.groupby("product")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 3 Products by Revenue:")
print(product_revenue.head(3))

Top 3 Products by Revenue:
product
Laptop    41535000
Tablet    19348000
Phone      8880000
Name: revenue, dtype: int64


In [17]:
# Calculate total units sold by product
product_units = (
    sales.groupby("product")["quantity"]
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 3 Products by Units Sold:")
print(product_units.head(3))


Top 3 Products by Units Sold:
product
Tablet    691
Laptop    639
Phone     592
Name: quantity, dtype: int64


In [18]:
product_comparison = pd.DataFrame({
    "Total Revenue (KES)": product_revenue,
    "Units Sold": product_units
})

display(product_comparison)

,Total Revenue (KES),Units Sold
product,,
Charger,642000,535
Headphones,1387500,555
Laptop,41535000,639
Phone,8880000,592
Tablet,19348000,691



### Business Interpretation

Laptops are the highest revenue-generating product, contributing KES 41,535,000, even though they rank second in units sold at 639 units. Tablets lead in sales volume with 691 units sold but rank second in revenue at KES 19,348,000. Phones rank third in both revenue and units sold, generating KES 8,880,000 from 592 units.

The difference between the rankings is mainly explained by variations in product prices. Laptops have a much higher unit price than tablets and phones, allowing them to generate more revenue despite lower sales volume than tablets.

Duka Electronics should continue prioritising laptops because of their strong revenue contribution while maintaining sufficient tablet stock to meet their higher customer demand.


## Task 5: Weekend versus Weekday Sales Analysis

This task compares sales performance between weekdays and weekends using order count, total revenue and average order revenue. The results will help identify whether customer purchasing behaviour differs depending on the day of the week.

In [19]:

# Group sales by weekend and weekday orders
weekend_comparison = sales.groupby("is_weekend").agg(
    number_of_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
)

# Give the groups meaningful labels
weekend_comparison.index = weekend_comparison.index.map({
    False: "Weekday",
    True: "Weekend"
})

print("Weekend vs Weekday Sales:")
display(weekend_comparison)


Weekend vs Weekday Sales:


,number_of_orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


In [20]:
# Calculate percentage shares
weekend_comparison["order_share_%"] = (
    weekend_comparison["number_of_orders"]
    / weekend_comparison["number_of_orders"].sum()
) * 100

weekend_comparison["revenue_share_%"] = (
    weekend_comparison["total_revenue"]
    / weekend_comparison["total_revenue"].sum()
) * 100

display(weekend_comparison.round(2))

,number_of_orders,total_revenue,average_order_revenue,order_share_%,revenue_share_%
is_weekend,,,,,
Weekday,723,51856200,71723.65,72.3,72.23
Weekend,277,19936300,71972.20,27.7,27.77



### Business Interpretation

Weekdays generate the majority of sales, accounting for 723 orders (72.3%) and KES 51,856,200 in revenue (72.23%). Weekend sales account for 277 orders (27.7%) and KES 19,936,300 in revenue (27.77%).

However, the average order revenue is slightly higher on weekends at KES 71,972.20 compared to KES 71,723.65 on weekdays, a difference of approximately KES 248.55.

This suggests that although fewer customers place orders during weekends, their average spending per order is marginally higher. Duka Electronics should maintain its weekday sales operations while exploring weekend promotions to increase customer traffic.


## Task 6: Monthly Revenue Performance

This task examines monthly revenue from January to June 2026 to identify the strongest and weakest sales months. The percentage difference between the best and worst months will help management understand the variation in monthly sales performance.

In [21]:

# Group revenue by month in calendar order
monthly_revenue = (
    sales.groupby(sales["order_date"].dt.month)["revenue"]
    .sum()
)

# Assign month names
monthly_revenue.index = pd.to_datetime(
    monthly_revenue.index, format="%m"
).month_name()

print("Monthly Revenue (KES):")
print(monthly_revenue)


Monthly Revenue (KES):
order_date
January     11786600
February    11772000
March       12396800
April       13518300
May         11415400
June        10903400
Name: revenue, dtype: int64


In [22]:
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

print("Best Month:", best_month)
print("Best Revenue: KES", f"{best_revenue:,.0f}")

print("\nWorst Month:", worst_month)
print("Worst Revenue: KES", f"{worst_revenue:,.0f}")

Best Month: April
Best Revenue: KES 13,518,300

Worst Month: June
Worst Revenue: KES 10,903,400


In [23]:
percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

print(
    f"Percentage Difference: {percentage_difference:.2f}%"
)

Percentage Difference: 23.98%



### Business Interpretation

April was the strongest sales month, generating KES 13,518,300 in revenue. June was the weakest month, generating KES 10,903,400. April's revenue was 23.98% higher than June's revenue, representing a difference of KES 2,614,900.

Revenue increased from January through April before declining in May and June. Management should investigate the factors that contributed to April's strong performance and identify possible reasons for the decline in the following months.

The analysis covers orders from 1 January to 29 June 2026, so June does not include the final day of the month.


## Task 7: Premium Online Sales in Nairobi and Mombasa

This task filters orders to identify Premium products sold through the Online channel in Nairobi and Mombasa. The analysis will establish the number of qualifying orders and their total revenue.

In [24]:

# Filter Premium Online orders from Nairobi and Mombasa
premium_online = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

# Calculate the number of qualifying orders
order_count = len(premium_online)

# Calculate total revenue
total_revenue = premium_online["revenue"].sum()

print("Number of qualifying orders:", order_count)
print(f"Total Revenue: KES {total_revenue:,.2f}")

display(premium_online.head())


Number of qualifying orders: 83
Total Revenue: KES 10,218,000.00


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


In [25]:
premium_online.to_csv(
    "premium_online_coast_capital.csv",
    index=False
)

print("Filtered dataset exported successfully!")

Filtered dataset exported successfully!


In [26]:
check = pd.read_csv("premium_online_coast_capital.csv")

print("Exported rows:", len(check))
print("Exported columns:", check.columns.tolist())
print("File exists:", __import__("os").path.exists(
    "premium_online_coast_capital.csv"
))

Exported rows: 83
Exported columns: ['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel', 'unit_price', 'revenue', 'month', 'weekday', 'is_weekend', 'price_band']
File exists: True


In [27]:
sales.to_csv("duka_sales_analysis.csv", index=False)

from google.colab import files
files.download("duka_sales_analysis.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## Task 8: Excel Spreadsheet Verification

The Duka Electronics branch revenue analysis was reproduced in Excel using SUMIFS to calculate total revenue for each branch. COUNTIFS was also used to calculate the number of orders per branch.

XLOOKUP was used to assign a region to each branch using a separate lookup table. The five branches were assigned to Capital, Coast, Western and Rift Valley regions.

A PivotTable was created to summarise total revenue by branch and sales channel, including In-store, Online and Phone order.

The Excel branch revenue figures matched the pandas results:

| Branch | Revenue (KES) |
|---|---:|
| Nairobi | 26,106,100 |
| Mombasa | 14,003,700 |
| Nakuru | 11,970,400 |
| Kisumu | 10,868,100 |
| Eldoret | 8,844,200 |
| **Grand Total** | **71,792,500** |

The PivotTable Grand Total was KES 71,792,500, matching the total revenue calculated in pandas. This confirms that the spreadsheet calculations and Python analysis produced consistent results.
